# Antibody form converter · 2.04
**Click ▶ below once**, then choose or drop one or multiple XLSX/CSV files.
The converter loads from GitHub when you start it. Download names retain your source filename and version.


In [ ]:
# @title ▶ Start converter { display-mode: "form" }
import hashlib
import io
from pathlib import Path
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from IPython.display import clear_output

VERSION = "2.04"
BUNDLE_URL = "https://raw.githubusercontent.com/sweber-maker/SW_tools/unified-v2.04/unified_converter/releases/unified_v2.04.zip"
BUNDLE_SHA256 = "e7b4a5db26be0bceb7d7c47c62b74cec68f833887095eaaf9fd0aca1b40bf68f"
BUNDLE_BYTES = 521868
print("Starting converter " + VERSION + "…")
try:
    with urllib.request.urlopen(BUNDLE_URL, timeout=30) as response:
        payload = response.read(BUNDLE_BYTES + 1)
    if hashlib.sha256(payload).hexdigest() != BUNDLE_SHA256:
        raise ValueError("Converter download did not match this release. Please reopen START_HERE.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "openpyxl==3.1.5", "lxml>=5,<7"],
        check=True, capture_output=True, text=True,
    )
    work_dir = Path(tempfile.mkdtemp(prefix="sw-tools-" + VERSION + "-"))
    with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
        for entry in bundle.infolist():
            path = Path(entry.filename)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError("Invalid converter package path.")
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Invalid converter package link.")
        bundle.extractall(work_dir)
    namespace = {"__name__": "sw_tools_launcher",
                 "__file__": str(work_dir / "launch.py")}
    exec(compile((work_dir / "launch.py").read_text(), "launch.py", "exec"), namespace)
    clear_output(wait=True)
    _engine, _controls = namespace["start"](work_dir)
except Exception as exc:
    print("Could not start the converter: " + str(exc))
    print("Please reopen START_HERE and run this cell again.")
    raise
